In [4]:
import pandas as pd
import numpy as np
import itertools
import time

# ===================== LOAD DATA =====================
data = pd.read_excel(
    '/Users/workk/Documents/[Think!]/[FMK]/Research Things/Books/Optimasi/Dataset/3-unit.xlsx',
    sheet_name='Table 3'
)
a = data['a'].values.astype(float)
b = data['b'].values.astype(float)
c = data['c'].values.astype(float)
Minimum_Capacity = data['pmin'].values.astype(float)
Maximum_Capacity = data['pmax'].values.astype(float)
ramp_up = data['rampup'].values.astype(float)
ramp_down = data['rampdown'].values.astype(float)
Unit = data['Unit'].values

# Optional current output as baseline for first step
Pnow = data['Pnow'].values.astype(float) if 'Pnow' in data.columns else None

power_demand_data = pd.read_excel('/Users/workk/Documents/[Think!]/[FMK]/Research Things/Books/Optimasi/Dataset/pd3u-6hours.xlsx')
power_demand = power_demand_data['load'].values.astype(float)

# ===================== DOA (UNCHANGED) =====================
def calculate_power(demand, a, b, c, pmin, pmax, rampup, rampdown,
                    pop=30, max_iter=100):
    dim = len(pmin)
    lb, ub = pmin, pmax

    # Objective: Fuel cost + penalty for demand mismatch
    def fobj(pos):
        fuel = np.sum(a + b * pos + c * pos**2)
        penalty = abs(demand - np.sum(pos)) * 10000
        return fuel + penalty

    # Initialization
    x = np.random.uniform(lb, ub, (pop, dim))
    fbest = np.inf
    sbest = np.ones(dim)
    sbestd = np.tile(np.ones(dim), (5, 1))
    fbestd = np.ones(5) * np.inf
    fbest_history = np.ones(max_iter)
    SELECT = np.arange(pop)

    # ---- Exploration phase (first 90% iterations) ----
    for it in range(int(0.9 * max_iter)):
        for m in range(5):
            # choose k with safe randint bounds
            low = max(1, int(np.ceil(dim / (8 * (m + 1)))))
            high_incl = max(2, int(np.ceil(dim / (3 * (m + 1)))))  # inclusive upper bound
            k = low if low >= high_incl else np.random.randint(low, high_incl + 1)

            start = int((m) * pop / 5)
            end = int((m + 1) * pop / 5)

            # Update group best
            for j in range(start, end):
                fit = fobj(x[j])
                if fit < fbestd[m]:
                    fbestd[m] = fit
                    sbestd[m] = x[j].copy()

            # Memory & forgetting/supplementation
            for j in range(start, end):
                x[j] = sbestd[m].copy()
                indices = np.random.permutation(dim)[:k]
                if np.random.rand() < 0.9:
                    for h in indices:
                        step = (np.random.rand() * (ub[h] - lb[h]) + lb[h]) * \
                               (np.cos((it + max_iter / 10) * np.pi / max_iter) + 1) / 2
                        x[j, h] += step
                        # Boundary handling
                        if x[j, h] > ub[h] or x[j, h] < lb[h]:
                            if dim > 15:
                                sel = np.random.choice(np.delete(SELECT, j))
                                x[j, h] = x[sel, h]
                            else:
                                x[j, h] = np.random.rand() * (ub[h] - lb[h]) + lb[h]
                else:
                    for h in indices:
                        x[j, h] = x[np.random.randint(pop), h]

            # Update global best
            if fbestd[m] < fbest:
                fbest = fbestd[m]
                sbest = sbestd[m].copy()

        fbest_history[it] = fbest

    # ---- Exploitation phase (last 10% iterations) ----
    for it in range(int(0.9 * max_iter), max_iter):
        for p in range(pop):
            fit = fobj(x[p])
            if fit < fbest:
                fbest = fit
                sbest = x[p].copy()

        # Local exploitation around sbest
        for j in range(pop):
            km = max(2, int(np.ceil(dim / 3)))
            # safe randint when km could be 2
            k = 2 if km == 2 else np.random.randint(2, km + 1)
            x[j] = sbest.copy()
            indices = np.random.permutation(dim)[:k]
            for h in indices:
                step = (np.random.rand() * (ub[h] - lb[h]) + lb[h]) * \
                       (np.cos(it * np.pi / max_iter) + 1) / 2
                x[j, h] += step
                if x[j, h] > ub[h] or x[j, h] < lb[h]:
                    if dim > 15:
                        sel = np.random.choice(np.delete(SELECT, j))
                        x[j, h] = x[sel, h]
                    else:
                        x[j, h] = np.random.rand() * (ub[h] - lb[h]) + lb[h]

        fbest_history[it] = fbest

    # Scale final best solution to meet demand
    total_power = np.sum(sbest)
    if total_power > 0:
        sbest *= demand / total_power
    sbest = np.clip(sbest, lb, ub)

    # Ramp rate adjustment (relative to x[0] as in your original code)
    diff = sbest - x[0]
    up_mask, down_mask = diff > 0, diff < 0
    sbest[up_mask] = x[0][up_mask] + np.minimum(diff[up_mask], rampup[up_mask])
    sbest[down_mask] = x[0][down_mask] + np.maximum(diff[down_mask], -rampdown[down_mask])

    return sbest, max_iter

# ===================== EXTRA HELPERS (AROUND DOA) =====================
def effective_bounds(baseline, lb, ub, rampup, rampdown):
    """Intersect UC bounds with ramp envelope from baseline."""
    if baseline is None:
        return lb.copy(), ub.copy()
    eff_lb = np.maximum(lb, baseline - rampdown)
    eff_ub = np.minimum(ub, baseline + rampup)
    eff_lb = np.minimum(eff_lb, eff_ub)
    return eff_lb.astype(float), eff_ub.astype(float)

def redistribute_exact(P, demand, eff_lb, eff_ub, tol=1e-6, max_pass=10):
    """Force sum(P)==demand within [eff_lb, eff_ub] by proportional allocation."""
    P = np.clip(P, eff_lb, eff_ub).astype(float)
    for _ in range(max_pass):
        diff = demand - P.sum()
        if abs(diff) <= tol:
            break
        if diff > 0:
            room = eff_ub - P
            total = room.sum()
            if total <= tol: break
            step = np.minimum(room, diff * (room / (total + 1e-12)))
            P += step
        else:
            room = P - eff_lb
            total = room.sum()
            if total <= tol: break
            step = np.minimum(room, (-diff) * (room / (total + 1e-12)))
            P -= step
    return np.clip(P, eff_lb, eff_ub)

def choose_commitment_feasible(demand, pmin, pmax, a, b, c, baseline, ru, rd):
    """
    Only used when demand < sum(pmin): pick ON subset that is ramp-feasible today.
    Chooses subset with feasible [eff_lb, eff_ub] and lowest heuristic cost.
    """
    n = len(pmin)
    candidates = []
    for r in range(1, n+1):
        for idxs in itertools.combinations(range(n), r):
            mask = np.zeros(n, dtype=bool); mask[list(idxs)] = True
            lb = pmin.copy(); ub = pmax.copy()
            lb[~mask] = 0.0; ub[~mask] = 0.0
            eff_lb, eff_ub = effective_bounds(baseline, lb, ub, ru, rd)
            smin, smax = eff_lb.sum(), eff_ub.sum()
            if smin <= demand <= smax:
                # heuristic: cost at lb + tiny marginal
                base = np.sum(a[mask] + b[mask]*lb[mask] + c[mask]*(lb[mask]**2))
                marg = np.sum(b[mask] + 2*c[mask]*lb[mask]) * 1e-3
                candidates.append((base+marg, mask))
    if candidates:
        candidates.sort(key=lambda x: x[0])
        return candidates[0][1]
    # fallback: if nothing ramp-feasible, keep all ON (we'll still match demand best we can)
    return np.ones(n, dtype=bool)

# ===================== COST =====================
def calculate_fuel_cost(P, a, b, c):
    squaredP = np.multiply(P, P)
    Fuel_Cost = np.add(a, np.add(np.multiply(b, P), np.multiply(c, squaredP)))
    total_Fuel_Cost = np.sum(Fuel_Cost)
    return Fuel_Cost, total_Fuel_Cost

# ===================== MAIN LOOP =====================
all_outputs = []
schedulling = []
summary_rows = []

baseline_prev = Pnow if Pnow is not None else None

for load_counter, demand in enumerate(power_demand, start=1):
    start_time = time.time()

    # ----- Unit commitment: only when demand < total pmin -----
    if demand < Minimum_Capacity.sum():
        on_mask = choose_commitment_feasible(
            demand, Minimum_Capacity, Maximum_Capacity,
            a, b, c, baseline_prev, ramp_up, ramp_down
        )
    else:
        on_mask = np.ones_like(Minimum_Capacity, dtype=bool)

    # UC bounds
    lb_uc = Minimum_Capacity.copy()
    ub_uc = Maximum_Capacity.copy()
    lb_uc[~on_mask] = 0.0
    ub_uc[~on_mask] = 0.0

    # Effective bounds (UC ∩ ramp envelope from baseline)
    eff_lb, eff_ub = effective_bounds(baseline_prev, lb_uc, ub_uc, ramp_up, ramp_down)

    # Safety: if still infeasible due to ramp, turn all ON and recompute envelope
    if not (eff_lb.sum() <= demand <= eff_ub.sum()):
        lb_uc = Minimum_Capacity.copy()
        ub_uc = Maximum_Capacity.copy()
        eff_lb, eff_ub = effective_bounds(baseline_prev, lb_uc, ub_uc, ramp_up, ramp_down)

    # ----- Call DOA UNCHANGED with effective bounds -----
    # Neutralize DOA's internal ramp tweak by sending very large ramp limits.
    huge = np.full_like(ramp_up, 1e12, dtype=float)
    P_raw, iterations = calculate_power(
        demand, a, b, c,
        eff_lb, eff_ub,          # use effective bounds as pmin/pmax
        huge, huge               # neutralize final ramp adjustment inside DOA
    )

    # ----- Enforce exact power balance inside effective bounds -----
    P = redistribute_exact(P_raw, demand, eff_lb, eff_ub, tol=1e-6, max_pass=10)

    # Costs
    Fuel_Cost, total_Fuel_Cost = calculate_fuel_cost(P, a, b, c)
    end_time = time.time()

    # Output record
    status = np.where(ub_uc > 0, "ON", "OFF")
    output = pd.DataFrame({'Unit': Unit, 'Status': status, 'Power Produced (MW)': P})
    print(f"Load Counter: {load_counter}")
    print(f"Demand: {demand:.2f} MW | Units ON: {np.sum(ub_uc>0)}/{len(ub_uc)}")
    print(output)
    print(f"Total Power Produced: {np.sum(P):.6f} MW")
    print(f"Total Fuel Cost: Rp {total_Fuel_Cost}")
    print(f'Computation Time: {end_time - start_time:.6f} s')
    print('------------------------------------------------\n')

    all_outputs.append({
        'Demand': demand,
        'Output': output,
        'Total Power Produced': float(np.sum(P)),
        'Total Fuel Cost': float(total_Fuel_Cost),
        'Computation Time': float(end_time - start_time)
    })

    summary_rows.append({
        'Load Counter': load_counter,
        'Demand (MW)': float(demand),
        'Units ON': int(np.sum(ub_uc>0)),
        'Total Power Produced (MW)': float(np.sum(P)),
        'Total Fuel Cost (kRp)': float(total_Fuel_Cost),
        'Computation Time (s)': float(end_time - start_time),
        'Iterations': int(iterations)
    })

    # Scheduling sheet (use per-unit cost, not total)
    info = {'Demand': demand}
    for unit_index, unit in enumerate(Unit):
        info[f'Unit {unit} Status'] = status[unit_index]
        info[f'Unit {unit} Power Produced'] = P[unit_index]
        info[f'Unit {unit} Cost'] = float(Fuel_Cost[unit_index])
    schedulling.append(info)

    # Baseline for next step (this already respects ramp envelope)
    baseline_prev = P.copy()

# ----- Ramp & limit checks (post) -----
ramp_rates, generation_limits = [], []
for i in range(1, len(all_outputs)):
    previous_output = all_outputs[i-1]['Output']['Power Produced (MW)'].values
    current_output = all_outputs[i]['Output']['Power Produced (MW)'].values
    ramp_rate = current_output - previous_output

    violations = np.logical_or(ramp_rate > ramp_up, ramp_rate < -ramp_down)
    ramp_rate_info = {'Demand': all_outputs[i]['Demand']}
    generation_limit_info = {'Demand': all_outputs[i]['Demand']}
    for unit_index, unit in enumerate(Unit):
        ramp_rate_info[f'Unit {unit}'] = float(ramp_rate[unit_index])
        ramp_rate_info[f'Unit {unit} Violation'] = bool(violations[unit_index])

        gen_violation = (current_output[unit_index] < 0) or \
                        (current_output[unit_index] > Maximum_Capacity[unit_index])
        generation_limit_info[f'Unit {unit}'] = float(current_output[unit_index])
        generation_limit_info[f'Unit {unit} Violation'] = bool(gen_violation)
    ramp_rates.append(ramp_rate_info)
    generation_limits.append(generation_limit_info)

# ===================== SAVE TO EXCEL =====================
output_file_path = '/Users/workk/Documents/[Think!]/[FMK]/Research Things/Books/Optimasi/DOA (ED-UC) 3u.xlsx'
with pd.ExcelWriter(output_file_path, engine='openpyxl') as writer:
    pd.DataFrame(schedulling).to_excel(writer, sheet_name='Schedulling', index=False)
    pd.DataFrame(summary_rows).to_excel(writer, sheet_name='Summary', index=False)
    pd.DataFrame(ramp_rates).to_excel(writer, sheet_name='Ramp Rates', index=False)
    pd.DataFrame(generation_limits).to_excel(writer, sheet_name='Generation Limits', index=False)

print("All results saved to 'DOA (ED-UC) 3u.xlsx' — DOA preserved, UC & exact balance applied.")

Load Counter: 1
Demand: 200.00 MW | Units ON: 1/3
   Unit Status  Power Produced (MW)
0     1    OFF                  0.0
1     2    OFF                  0.0
2     3     ON                200.0
Total Power Produced: 200.000000 MW
Total Fuel Cost: Rp 2735.8
Computation Time: 0.022114 s
------------------------------------------------

Load Counter: 2
Demand: 230.00 MW | Units ON: 2/3
   Unit Status  Power Produced (MW)
0     1    OFF                  0.0
1     2     ON                 80.0
2     3     ON                150.0
Total Power Produced: 230.000000 MW
Total Fuel Cost: Rp 2893.366
Computation Time: 0.022619 s
------------------------------------------------

Load Counter: 3
Demand: 250.00 MW | Units ON: 3/3
   Unit Status  Power Produced (MW)
0     1     ON            80.000000
1     2     ON           104.765208
2     3     ON            65.234792
Total Power Produced: 250.000000 MW
Total Fuel Cost: Rp 2976.729814076256
Computation Time: 0.021735 s
-----------------------------

In [5]:
import pandas as pd
import numpy as np
import itertools
import time

# ===================== LOAD DATA =====================
data = pd.read_excel(
    '/Users/workk/Documents/[Think!]/[FMK]/Research Things/Books/Optimasi/Dataset/15-unit.xlsx',
    sheet_name='Table 2'
)
a = data['a'].values.astype(float)
b = data['b'].values.astype(float)
c = data['c'].values.astype(float)
Minimum_Capacity = data['pmin'].values.astype(float)
Maximum_Capacity = data['pmax'].values.astype(float)
ramp_up = data['rampup'].values.astype(float)
ramp_down = data['rampdown'].values.astype(float)
Unit = data['Unit'].values

# Optional current output as baseline for first step
Pnow = data['Pnow'].values.astype(float) if 'Pnow' in data.columns else None

power_demand_data = pd.read_excel('/Users/workk/Documents/[Think!]/[FMK]/Research Things/Books/Optimasi/Dataset/pd15u-6hours.xlsx')
power_demand = power_demand_data['load'].values.astype(float)

# ===================== DOA (UNCHANGED) =====================
def calculate_power(demand, a, b, c, pmin, pmax, rampup, rampdown,
                    pop=30, max_iter=100):
    dim = len(pmin)
    lb, ub = pmin, pmax

    # Objective: Fuel cost + penalty for demand mismatch
    def fobj(pos):
        fuel = np.sum(a + b * pos + c * pos**2)
        penalty = abs(demand - np.sum(pos)) * 10000
        return fuel + penalty

    # Initialization
    x = np.random.uniform(lb, ub, (pop, dim))
    fbest = np.inf
    sbest = np.ones(dim)
    sbestd = np.tile(np.ones(dim), (5, 1))
    fbestd = np.ones(5) * np.inf
    fbest_history = np.ones(max_iter)
    SELECT = np.arange(pop)

    # ---- Exploration phase (first 90% iterations) ----
    for it in range(int(0.9 * max_iter)):
        for m in range(5):
            # choose k with safe randint bounds
            low = max(1, int(np.ceil(dim / (8 * (m + 1)))))
            high_incl = max(2, int(np.ceil(dim / (3 * (m + 1)))))  # inclusive upper bound
            k = low if low >= high_incl else np.random.randint(low, high_incl + 1)

            start = int((m) * pop / 5)
            end = int((m + 1) * pop / 5)

            # Update group best
            for j in range(start, end):
                fit = fobj(x[j])
                if fit < fbestd[m]:
                    fbestd[m] = fit
                    sbestd[m] = x[j].copy()

            # Memory & forgetting/supplementation
            for j in range(start, end):
                x[j] = sbestd[m].copy()
                indices = np.random.permutation(dim)[:k]
                if np.random.rand() < 0.9:
                    for h in indices:
                        step = (np.random.rand() * (ub[h] - lb[h]) + lb[h]) * \
                               (np.cos((it + max_iter / 10) * np.pi / max_iter) + 1) / 2
                        x[j, h] += step
                        # Boundary handling
                        if x[j, h] > ub[h] or x[j, h] < lb[h]:
                            if dim > 15:
                                sel = np.random.choice(np.delete(SELECT, j))
                                x[j, h] = x[sel, h]
                            else:
                                x[j, h] = np.random.rand() * (ub[h] - lb[h]) + lb[h]
                else:
                    for h in indices:
                        x[j, h] = x[np.random.randint(pop), h]

            # Update global best
            if fbestd[m] < fbest:
                fbest = fbestd[m]
                sbest = sbestd[m].copy()

        fbest_history[it] = fbest

    # ---- Exploitation phase (last 10% iterations) ----
    for it in range(int(0.9 * max_iter), max_iter):
        for p in range(pop):
            fit = fobj(x[p])
            if fit < fbest:
                fbest = fit
                sbest = x[p].copy()

        # Local exploitation around sbest
        for j in range(pop):
            km = max(2, int(np.ceil(dim / 3)))
            # safe randint when km could be 2
            k = 2 if km == 2 else np.random.randint(2, km + 1)
            x[j] = sbest.copy()
            indices = np.random.permutation(dim)[:k]
            for h in indices:
                step = (np.random.rand() * (ub[h] - lb[h]) + lb[h]) * \
                       (np.cos(it * np.pi / max_iter) + 1) / 2
                x[j, h] += step
                if x[j, h] > ub[h] or x[j, h] < lb[h]:
                    if dim > 15:
                        sel = np.random.choice(np.delete(SELECT, j))
                        x[j, h] = x[sel, h]
                    else:
                        x[j, h] = np.random.rand() * (ub[h] - lb[h]) + lb[h]

        fbest_history[it] = fbest

    # Scale final best solution to meet demand
    total_power = np.sum(sbest)
    if total_power > 0:
        sbest *= demand / total_power
    sbest = np.clip(sbest, lb, ub)

    # Ramp rate adjustment (relative to x[0] as in your original code)
    diff = sbest - x[0]
    up_mask, down_mask = diff > 0, diff < 0
    sbest[up_mask] = x[0][up_mask] + np.minimum(diff[up_mask], rampup[up_mask])
    sbest[down_mask] = x[0][down_mask] + np.maximum(diff[down_mask], -rampdown[down_mask])

    return sbest, max_iter

# ===================== EXTRA HELPERS (AROUND DOA) =====================
def effective_bounds(baseline, lb, ub, rampup, rampdown):
    """Intersect UC bounds with ramp envelope from baseline."""
    if baseline is None:
        return lb.copy(), ub.copy()
    eff_lb = np.maximum(lb, baseline - rampdown)
    eff_ub = np.minimum(ub, baseline + rampup)
    eff_lb = np.minimum(eff_lb, eff_ub)
    return eff_lb.astype(float), eff_ub.astype(float)

def redistribute_exact(P, demand, eff_lb, eff_ub, tol=1e-6, max_pass=10):
    """Force sum(P)==demand within [eff_lb, eff_ub] by proportional allocation."""
    P = np.clip(P, eff_lb, eff_ub).astype(float)
    for _ in range(max_pass):
        diff = demand - P.sum()
        if abs(diff) <= tol:
            break
        if diff > 0:
            room = eff_ub - P
            total = room.sum()
            if total <= tol: break
            step = np.minimum(room, diff * (room / (total + 1e-12)))
            P += step
        else:
            room = P - eff_lb
            total = room.sum()
            if total <= tol: break
            step = np.minimum(room, (-diff) * (room / (total + 1e-12)))
            P -= step
    return np.clip(P, eff_lb, eff_ub)

def choose_commitment_feasible(demand, pmin, pmax, a, b, c, baseline, ru, rd):
    """
    Only used when demand < sum(pmin): pick ON subset that is ramp-feasible today.
    Chooses subset with feasible [eff_lb, eff_ub] and lowest heuristic cost.
    """
    n = len(pmin)
    candidates = []
    for r in range(1, n+1):
        for idxs in itertools.combinations(range(n), r):
            mask = np.zeros(n, dtype=bool); mask[list(idxs)] = True
            lb = pmin.copy(); ub = pmax.copy()
            lb[~mask] = 0.0; ub[~mask] = 0.0
            eff_lb, eff_ub = effective_bounds(baseline, lb, ub, ru, rd)
            smin, smax = eff_lb.sum(), eff_ub.sum()
            if smin <= demand <= smax:
                # heuristic: cost at lb + tiny marginal
                base = np.sum(a[mask] + b[mask]*lb[mask] + c[mask]*(lb[mask]**2))
                marg = np.sum(b[mask] + 2*c[mask]*lb[mask]) * 1e-3
                candidates.append((base+marg, mask))
    if candidates:
        candidates.sort(key=lambda x: x[0])
        return candidates[0][1]
    # fallback: if nothing ramp-feasible, keep all ON (we'll still match demand best we can)
    return np.ones(n, dtype=bool)

# ===================== COST =====================
def calculate_fuel_cost(P, a, b, c):
    squaredP = np.multiply(P, P)
    Fuel_Cost = np.add(a, np.add(np.multiply(b, P), np.multiply(c, squaredP)))
    total_Fuel_Cost = np.sum(Fuel_Cost)
    return Fuel_Cost, total_Fuel_Cost

# ===================== MAIN LOOP =====================
all_outputs = []
schedulling = []
summary_rows = []

baseline_prev = Pnow if Pnow is not None else None

for load_counter, demand in enumerate(power_demand, start=1):
    start_time = time.time()

    # ----- Unit commitment: only when demand < total pmin -----
    if demand < Minimum_Capacity.sum():
        on_mask = choose_commitment_feasible(
            demand, Minimum_Capacity, Maximum_Capacity,
            a, b, c, baseline_prev, ramp_up, ramp_down
        )
    else:
        on_mask = np.ones_like(Minimum_Capacity, dtype=bool)

    # UC bounds
    lb_uc = Minimum_Capacity.copy()
    ub_uc = Maximum_Capacity.copy()
    lb_uc[~on_mask] = 0.0
    ub_uc[~on_mask] = 0.0

    # Effective bounds (UC ∩ ramp envelope from baseline)
    eff_lb, eff_ub = effective_bounds(baseline_prev, lb_uc, ub_uc, ramp_up, ramp_down)

    # Safety: if still infeasible due to ramp, turn all ON and recompute envelope
    if not (eff_lb.sum() <= demand <= eff_ub.sum()):
        lb_uc = Minimum_Capacity.copy()
        ub_uc = Maximum_Capacity.copy()
        eff_lb, eff_ub = effective_bounds(baseline_prev, lb_uc, ub_uc, ramp_up, ramp_down)

    # ----- Call DOA UNCHANGED with effective bounds -----
    # Neutralize DOA's internal ramp tweak by sending very large ramp limits.
    huge = np.full_like(ramp_up, 1e12, dtype=float)
    P_raw, iterations = calculate_power(
        demand, a, b, c,
        eff_lb, eff_ub,          # use effective bounds as pmin/pmax
        huge, huge               # neutralize final ramp adjustment inside DOA
    )

    # ----- Enforce exact power balance inside effective bounds -----
    P = redistribute_exact(P_raw, demand, eff_lb, eff_ub, tol=1e-6, max_pass=10)

    # Costs
    Fuel_Cost, total_Fuel_Cost = calculate_fuel_cost(P, a, b, c)
    end_time = time.time()

    # Output record
    status = np.where(ub_uc > 0, "ON", "OFF")
    output = pd.DataFrame({'Unit': Unit, 'Status': status, 'Power Produced (MW)': P})
    print(f"Load Counter: {load_counter}")
    print(f"Demand: {demand:.2f} MW | Units ON: {np.sum(ub_uc>0)}/{len(ub_uc)}")
    print(output)
    print(f"Total Power Produced: {np.sum(P):.6f} MW")
    print(f"Total Fuel Cost: Rp {total_Fuel_Cost}")
    print(f'Computation Time: {end_time - start_time:.6f} s')
    print('------------------------------------------------\n')

    all_outputs.append({
        'Demand': demand,
        'Output': output,
        'Total Power Produced': float(np.sum(P)),
        'Total Fuel Cost': float(total_Fuel_Cost),
        'Computation Time': float(end_time - start_time)
    })

    summary_rows.append({
        'Load Counter': load_counter,
        'Demand (MW)': float(demand),
        'Units ON': int(np.sum(ub_uc>0)),
        'Total Power Produced (MW)': float(np.sum(P)),
        'Total Fuel Cost (kRp)': float(total_Fuel_Cost),
        'Computation Time (s)': float(end_time - start_time),
        'Iterations': int(iterations)
    })

    # Scheduling sheet (use per-unit cost, not total)
    info = {'Demand': demand}
    for unit_index, unit in enumerate(Unit):
        info[f'Unit {unit} Status'] = status[unit_index]
        info[f'Unit {unit} Power Produced'] = P[unit_index]
        info[f'Unit {unit} Cost'] = float(Fuel_Cost[unit_index])
    schedulling.append(info)

    # Baseline for next step (this already respects ramp envelope)
    baseline_prev = P.copy()

# ----- Ramp & limit checks (post) -----
ramp_rates, generation_limits = [], []
for i in range(1, len(all_outputs)):
    previous_output = all_outputs[i-1]['Output']['Power Produced (MW)'].values
    current_output = all_outputs[i]['Output']['Power Produced (MW)'].values
    ramp_rate = current_output - previous_output

    violations = np.logical_or(ramp_rate > ramp_up, ramp_rate < -ramp_down)
    ramp_rate_info = {'Demand': all_outputs[i]['Demand']}
    generation_limit_info = {'Demand': all_outputs[i]['Demand']}
    for unit_index, unit in enumerate(Unit):
        ramp_rate_info[f'Unit {unit}'] = float(ramp_rate[unit_index])
        ramp_rate_info[f'Unit {unit} Violation'] = bool(violations[unit_index])

        gen_violation = (current_output[unit_index] < 0) or \
                        (current_output[unit_index] > Maximum_Capacity[unit_index])
        generation_limit_info[f'Unit {unit}'] = float(current_output[unit_index])
        generation_limit_info[f'Unit {unit} Violation'] = bool(gen_violation)
    ramp_rates.append(ramp_rate_info)
    generation_limits.append(generation_limit_info)

# ===================== SAVE TO EXCEL =====================
output_file_path = '/Users/workk/Documents/[Think!]/[FMK]/Research Things/Books/Optimasi/DOA (ED-UC) 15u.xlsx'
with pd.ExcelWriter(output_file_path, engine='openpyxl') as writer:
    pd.DataFrame(schedulling).to_excel(writer, sheet_name='Schedulling', index=False)
    pd.DataFrame(summary_rows).to_excel(writer, sheet_name='Summary', index=False)
    pd.DataFrame(ramp_rates).to_excel(writer, sheet_name='Ramp Rates', index=False)
    pd.DataFrame(generation_limits).to_excel(writer, sheet_name='Generation Limits', index=False)

print("All results saved to 'DOA (ED-UC) 15u.xlsx' — DOA preserved, UC & exact balance applied.")

Load Counter: 1
Demand: 800.00 MW | Units ON: 4/15
    Unit Status  Power Produced (MW)
0      1    OFF             0.000000
1      2    OFF             0.000000
2      3    OFF             0.000000
3      4    OFF             0.000000
4      5    OFF             0.000000
5      6    OFF             0.000000
6      7     ON           411.319987
7      8    OFF             0.000000
8      9     ON           158.338787
9     10     ON           155.055056
10    11     ON            75.286170
11    12    OFF             0.000000
12    13    OFF             0.000000
13    14    OFF             0.000000
14    15    OFF             0.000000
Total Power Produced: 800.000000 MW
Total Fuel Cost: Rp 13842.401865077405
Computation Time: 0.305852 s
------------------------------------------------

Load Counter: 2
Demand: 850.00 MW | Units ON: 4/15
    Unit Status  Power Produced (MW)
0      1    OFF             0.000000
1      2    OFF             0.000000
2      3    OFF             0.000000
3   